In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/muratkokludataset/dry-bean-dataset/Dry_Bean_Dataset/Dry_Bean_Dataset.arff
/kaggle/input/datasets/muratkokludataset/dry-bean-dataset/Dry_Bean_Dataset/Dry_Bean_Dataset.xlsx
/kaggle/input/datasets/muratkokludataset/dry-bean-dataset/Dry_Bean_Dataset/Dry_Bean_Dataset_Citation_Request.txt


In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/muratkokludataset/dry-bean-dataset/Dry_Bean_Dataset/Dry_Bean_Dataset.arff
/kaggle/input/datasets/muratkokludataset/dry-bean-dataset/Dry_Bean_Dataset/Dry_Bean_Dataset.xlsx
/kaggle/input/datasets/muratkokludataset/dry-bean-dataset/Dry_Bean_Dataset/Dry_Bean_Dataset_Citation_Request.txt


In [3]:

import pandas as pd

file_path = '/kaggle/input/datasets/muratkokludataset/dry-bean-dataset/Dry_Bean_Dataset/Dry_Bean_Dataset.arff'

df = pd.read_csv(
    file_path,
    comment='%',
    header=None
)

df = df.dropna(how='all')

print(df.head())


ParserError: Error tokenizing data. C error: Expected 1 fields in line 27, saw 7


In [ ]:

import pandas as pd
from scipy.io import arff

file_path = '/kaggle/input/datasets/muratkokludataset/dry-bean-dataset/Dry_Bean_Dataset/Dry_Bean_Dataset.arff'

data, meta = arff.loadarff(file_path)

df = pd.DataFrame(data)

# Convert text columns from bytes to strings
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.decode('utf-8')

print(df.head())


In [ ]:

print("Dataset Shape:", df.shape)

print("\nNumber of Classes:")
print(df['Class'].nunique())

print("\nClass Distribution:")
print(df['Class'].value_counts())

print("\nMissing Values:")
print(df.isnull().sum().sum())


In [ ]:

print("Dataset Shape:", df.shape)

print("\nDataset Information:")
df.info()

print("\nClass Distribution:")
print(df['Class'].value_counts())

print("\nMissing Values:")
print(df.isnull().sum().sum())


In [ ]:

X = df.drop('Class', axis=1).values
y = df['Class'].values

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)


In [ ]:

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

print("Classes:", label_encoder.classes_)
print("Encoded Values:", np.unique(y))


In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


In [ ]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Scaling Completed Successfully!")


In [ ]:

num_classes = len(np.unique(y_train))

Y_train = np.eye(num_classes)[y_train]

print("Number of Classes:", num_classes)
print("Y_train Shape:", Y_train.shape)
print(Y_train[:5])


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

n_samples, n_features = X_train.shape

W = np.random.randn(n_features, num_classes) * 0.01
b = np.zeros((1, num_classes))

print("Weights Shape:", W.shape)
print("Bias Shape:", b.shape)


In [ ]:

def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


In [ ]:

def compute_loss(Y_true, Y_pred):
    epsilon = 1e-15
    Y_pred = np.clip(Y_pred, epsilon, 1 - epsilon)

    loss = -np.mean(
        np.sum(Y_true * np.log(Y_pred), axis=1)
    )

    return loss


In [ ]:

learning_rate = 0.05
epochs = 1000
beta = 0.9

vW = np.zeros_like(W)
vb = np.zeros_like(b)

loss_history = []

for epoch in range(epochs):

    # Forward Propagation
    z = np.dot(X_train, W) + b
    y_pred = softmax(z)

    # Calculate Loss
    loss = compute_loss(Y_train, y_pred)
    loss_history.append(loss)

    # Calculate Gradient
    error = y_pred - Y_train

    dW = np.dot(X_train.T, error) / n_samples
    db = np.sum(error, axis=0, keepdims=True) / n_samples

    # Momentum
    vW = beta * vW + dW
    vb = beta * vb + db

    # Update Parameters
    W -= learning_rate * vW
    b -= learning_rate * vb

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

print("Training Completed!")


In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(loss_history)

plt.title("GD with Momentum - Loss Curve")
plt.xlabel("Epochs")
plt.ylabel("Loss")

plt.grid()
plt.show()


In [ ]:

z_test = np.dot(X_test, W) + b

y_test_prob = softmax(z_test)

y_pred = np.argmax(y_test_prob, axis=1)

print("Predictions:", y_pred[:10])
print("Actual:", y_test[:10])


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test, y_pred, average='macro'
)

recall = recall_score(
    y_test, y_pred, average='macro'
)

f1 = f1_score(
    y_test, y_pred, average='macro'
)

print("GD with Momentum Results")
print("-------------------------")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


In [ ]:

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.classes_
)

fig, ax = plt.subplots(figsize=(10, 8))

disp.plot(
    cmap='Blues',
    xticks_rotation=45,
    ax=ax
)

plt.title("GD with Momentum - Confusion Matrix")
plt.show()


In [ ]:

gd_results = {
    'Optimizer': 'GD with Momentum',
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1 Score': f1,
    'Loss': loss_history[-1]
}

results_df = pd.DataFrame([gd_results])

print(results_df)
